# Examen blanc B : GRU ou encodeur Transformer pour reconnaitre une activite ?

**Tache** : donnees du TP4 (Human Activity Recognition), series temporelles 128 pas x 9 capteurs,
6 classes d'activite.

**Questions** :
1. Un encodeur Transformer (self-attention) fait-il mieux, moins bien ou pareil qu'un GRU,
   a budget comparable ?
2. L'encodage de position est-il necessaire pour le Transformer ? (sans lui, l'attention
   voit la sequence comme un **ensemble** : l'ordre des pas de temps est perdu)

**Modeles** (memes splits, meme optimiseur, memes epochs, 3 seeds) :
- `gru` : GRU 1 couche, 150 unites (~73k parametres, comme le Transformer), on classe a partir de h_n (comme le LSTM du TP4) ;
- `transformer` : projection lineaire 9 -> 64, + encodage de position appris, + token [CLS],
  2 couches d'encodeur, classification sur le vecteur de [CLS] (comme BERT) ;
- `transformer_nopos` : identique, sans encodage de position (ablation).

In [ ]:
import os, time, random, copy
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")
QUICK = False  # True pour un test tres rapide (CPU)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## 1. Donnees (fonction `load_dataset` du TP4)

In [ ]:
def load_dataset(train, path='data'):
    import zipfile
    zip1 = os.path.join(path, 'human+activity+recognition+using+smartphones.zip')
    if not os.path.isfile(zip1):
        from urllib.request import urlretrieve
        os.makedirs(path, exist_ok=True)
        urlretrieve('https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip', zip1)
    if not os.path.isfile(os.path.join(path, 'UCI HAR Dataset.zip')):
        with zipfile.ZipFile(zip1, 'r') as z:
            z.extractall(path)
    if not os.path.isdir(os.path.join(path, 'UCI HAR Dataset')):
        with zipfile.ZipFile(os.path.join(path, 'UCI HAR Dataset.zip'), 'r') as z:
            z.extractall(path)
    set_ = 'train' if train else 'test'
    names = [f'{s}_{a}_{set_}.txt' for s in ('body_acc', 'body_gyro', 'total_acc') for a in 'xyz']
    X = np.asarray([np.loadtxt(os.path.join(path, 'UCI HAR Dataset', set_, 'Inertial Signals', f)) for f in names])
    X = np.transpose(X, (1, 2, 0))                                   # (n, 128, 9)
    y = np.loadtxt(os.path.join(path, 'UCI HAR Dataset', set_, f'y_{set_}.txt'))
    subject = np.loadtxt(os.path.join(path, 'UCI HAR Dataset', set_, f'subject_{set_}.txt'), dtype='int64')
    return X, y, subject


X_train, y_train, subject_train = load_dataset(train=True)
X_test, y_test, subject_test = load_dataset(train=False)
print(X_train.shape, X_test.shape)

**Split de validation par sujet.** Le test contient des personnes jamais vues. Pour que la
validation mesure la meme chose (generaliser a de nouvelles personnes), on met de cote
5 sujets entiers du train, plutot qu'un `random_split` ligne a ligne : des fenetres d'une
meme personne qui se chevauchent a 50 % seraient sinon a la fois en train et en validation.

In [ ]:
subjects = np.unique(subject_train)
val_subjects = np.random.RandomState(0).choice(subjects, 5, replace=False)
is_val = np.isin(subject_train, val_subjects)
print("Sujets de validation :", sorted(val_subjects), "| fenetres :", is_val.sum())

# Standardisation par canal, avec les statistiques du train seulement (pas de fuite)
mean = X_train[~is_val].mean(axis=(0, 1), keepdims=True)
std = X_train[~is_val].std(axis=(0, 1), keepdims=True)


def to_tensors(X, y):
    X = torch.from_numpy((X - mean) / std).to(dtype=torch.float32)
    y = torch.from_numpy(y - 1).to(dtype=torch.int64)               # classes 1..6 -> 0..5
    return X, y


class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_data = CustomDataset(*to_tensors(X_train[~is_val], y_train[~is_val]))
val_data = CustomDataset(*to_tensors(X_train[is_val], y_train[is_val]))
test_data = CustomDataset(*to_tensors(X_test, y_test))
val_dataloader = DataLoader(val_data, batch_size=256)
test_dataloader = DataLoader(test_data, batch_size=256)

## 2. Modeles

In [ ]:
class Classifier(nn.Module):
    """Partie commune : optimiseur et pas d'entrainement/validation (design du TP4)."""

    def __init__(self):
        super().__init__()
        self.loss_fn_mean = nn.CrossEntropyLoss(reduction='mean')
        self.loss_fn_sum = nn.CrossEntropyLoss(reduction='sum')

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

    def training_step(self, batch):
        # TODO
        ...

    def validation_step(self, batch):
        # TODO: (loss sommee, classes predites, vrais labels), sans gradient
        ...


class GRUNetwork(Classifier):
    def __init__(self, n_channels=9, n_classes=6, n_hidden=150):
        super().__init__()
        # TODO
        ...

    def forward(self, x):                       # x : (batch, 128, 9)
        # TODO: classer a partir de h_n (comme le LSTM du TP4)
        ...


class TransformerNetwork(Classifier):
    def __init__(self, n_channels=9, n_classes=6, d_model=64, n_heads=4, n_layers=2,
                 seq_len=128, use_position=True):
        super().__init__()
        # TODO: projection 9 -> d_model ; token [CLS] appris (nn.Parameter) ;
        #       encodage de position appris (nn.Parameter de forme (1, seq_len + 1, d_model)) ;
        #       nn.TransformerEncoder(nn.TransformerEncoderLayer(..., batch_first=True), n_layers) ;
        #       Linear d_model -> n_classes.
        ...

    def forward(self, x):                       # x : (batch, 128, 9)
        # TODO: projeter, ajouter [CLS] en tete, ajouter la position si use_position,
        #       encoder, classer a partir de la sortie en position 0.
        ...


def make_model(name):
    return {"gru": lambda: GRUNetwork(),
            "transformer": lambda: TransformerNetwork(use_position=True),
            "transformer_nopos": lambda: TransformerNetwork(use_position=False)}[name]()

## 3. Entrainement (Trainer du TP4, avec historique et meilleure epoch en validation)

In [ ]:
def evaluate(dataloader, model):
    # TODO: (loss moyenne, accuracy) en mode eval
    ...


class Trainer():
    def __init__(self, max_epochs):
        self.max_epochs = max_epochs

    def fit(self, model, train_dataloader, val_dataloader):
        # TODO: boucle du TP4 + history + meilleure epoch en validation
        ...


def run(name, seed, max_epochs):
    # TODO: seed, DataLoader (generator seede), modele, entrainement ;
    #       renvoyer (model, history, n_params, secondes)
    ...

## 4. Grille : 3 modeles x 3 seeds, memes epochs

In [ ]:
MODELS = ["gru", "transformer", "transformer_nopos"]
SEEDS = [0] if QUICK else [0, 1, 2]
EPOCHS = 1 if QUICK else 15

results, histories = [], {}
for name in MODELS:
    for seed in SEEDS:
        model, history, n_params, secs = run(name, seed, EPOCHS)
        histories[(name, seed)] = history
        results.append((name, seed, max(history["val_acc"]), n_params, secs))
        print(f"{name:18s} seed={seed} best val_acc={max(history['val_acc']):.3f} "
              f"params={n_params:,} ({secs:.0f} s)")

In [ ]:
import pandas as pd

df = pd.DataFrame(results, columns=["model", "seed", "val_acc", "n_params", "seconds"])
print(df.groupby("model").agg(mean=("val_acc", "mean"), std=("val_acc", "std"),
                              n_params=("n_params", "first"), seconds=("seconds", "mean")).round(3))

fig, axes = plt.subplots(1, len(MODELS), figsize=(14, 3.5), sharey=True)
for ax, name in zip(axes, MODELS):
    for seed in SEEDS:
        h = histories[(name, seed)]
        ax.plot(h["train_loss"], color="C0", alpha=.7, label="train" if seed == 0 else None)
        ax.plot(h["val_loss"], color="C1", alpha=.7, label="val" if seed == 0 else None)
    ax.set_title(name); ax.set_xlabel("epoch")
axes[0].set_ylabel("loss"); axes[0].legend(); plt.show()

## 5. Test : une seule fois, a la fin

In [ ]:
for name in MODELS:
    model, _, _, _ = run(name, seed=0, max_epochs=EPOCHS)
    _, test_acc = evaluate(test_dataloader, model)
    print(f"TEST {name:18s}: accuracy {100 * test_acc:.1f}%")